# Block 02: Data, Preprocessing, and PSD Audit

**Goal**  
Freeze the real-data analysis contract used by later K-alpha and mixed real/simulation studies.

**Checklist alignment**  
Supports the shared preprocessing assumptions behind E5 and E12.

**Outputs**  
- tables under `results/tables/`
- figures under `results/figures/`
- manifests under `results/manifests/`
- executed notebook copy under `results/notebooks/` when this suite is run with `--execute`

In [1]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import pandas as pd

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "experiment_checklist.md").exists() and (candidate / "implementation").exists():
            return candidate
    raise RuntimeError("Could not locate repo root for notebook execution.")

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from implementation.notebook_support import *

cfg = CanonicalConfig().validate()
dirs = ensure_results_dirs(cfg)
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

In [2]:
out = run_block02_audit(cfg)
bundle = out["bundle"]
audit_df = out["audit_df"]
reports_df = out["reports_df"]
display(audit_df)
display(reports_df)

,metric,value
0,n_events,4.358000e+03
1,trace_len,3.276800e+04
2,baseline_before_mean,1.310263e+02
3,baseline_before_std,1.130269e+00
4,baseline_after_mean,-2.468084e-16
5,baseline_after_std,8.173377e-15
6,canonical_psd_mean,1.976002e-05
7,empirical_pretrigger_psd_mean,2.446796e-04
8,weight_min,6.968760e+01
9,weight_max,1.141692e+06


,trace_path,rq_path,template_path,psd_path
0,data/k_alpha_traces.h5,data/k_alpha_rqs.h5,data/template_K_alpha_tight.npy,data/weight/noise_psd_pink.npy


In [3]:
fig, ax = plt.subplots(figsize=(6, 4))
empirical = bundle.metadata["empirical_psd"]
freqs = bundle.metadata["empirical_psd_freqs"]
ax.loglog(freqs[1:], bundle.psd_one_sided[1:], label="canonical PSD")
ax.loglog(freqs[1:], empirical[1:], label="empirical pretrigger PSD")
ax.set_xlabel("frequency [Hz]")
ax.set_ylabel("PSD")
ax.set_title("Real-data PSD audit")
ax.legend()
save_figure(fig, dirs["figures"] / "block_02_psd_audit.png")
plt.close(fig)

In [4]:
audit_path = dirs["tables"] / "block_02_psd_audit.csv"
reports_path = dirs["tables"] / "block_02_data_sources.csv"
manifest_path = dirs["manifests"] / "block_02_split_manifest.json"
save_dataframe(audit_df, audit_path)
save_dataframe(reports_df, reports_path)
save_json(out["split_manifest"], manifest_path)
pd.DataFrame(
    [
        manifest_row("block_02", "real-support", str(audit_path.relative_to(REPO_ROOT)), cfg, bundle),
        manifest_row("block_02", "real-support", str(manifest_path.relative_to(REPO_ROOT)), cfg, bundle),
    ]
)

,block_id,regime,output_path,seed,trace_len,pretrigger,data_source,n_events,template_path,psd_path
0,block_02,real-support,results/tables/block_02_psd_audit.csv,314159,32768,4000,CAL-kalpha,4358,data/template_K_alpha_tight.npy,data/weight/noise_psd_pink.npy
1,block_02,real-support,results/manifests/block_02_split_manifest.json,314159,32768,4000,CAL-kalpha,4358,data/template_K_alpha_tight.npy,data/weight/noise_psd_pink.npy
